# Quantum Mile: six junctions, straight-through cars

This notebook reproduces the first modeled signal decision and reads any recorded IBM result from the accompanying evidence. Run All uses local Qiskit and saved files.

In [1]:
from pathlib import Path
import json
from quantum_core import circuit, verify, analysis, sha256

root = Path.cwd()
fixture = root / 'fixtures.json'
data = json.loads(fixture.read_text())
case, = data['cases']
assert case['id'] == 'network-balanced-open-through-t0'
assert case['context']['input']['routing'] == 'through'
for name, expected in data['sourceSha256'].items():
    assert sha256(root / 'source' / name) == expected
check = verify(case)
print('Fixture SHA-256:', sha256(fixture))
print('Qubits:', check['qubits'])
print('Verified circuit states:', check['frames'])
print('Maximum probability error:', check['maximumProbabilityError'])
print(circuit(case, measured=True).draw('text', fold=94))

Fixture SHA-256: b33afd85f39ad9b59ceb6e96291effde96116fcfe578af09b306a710aefc4abb
Qubits: 6
Verified circuit states: 28
Maximum probability error: 1.1102230246251565e-16
        ┌───┐ ┌────────────┐                                                             »
   q_0: ┤ H ├─┤ Rz(1.3971) ├───■────────────────────────────■──────────────■─────────────»
        ├───┤┌┴────────────┴┐  │ZZ(-0.1572)                 │              │             »
   q_1: ┤ H ├┤ Rz(-0.56868) ├──■─────────────■──────────────┼──────────────┼─────────────»
        ├───┤└┬────────────┬┘                │ZZ(-0.18845)  │              │             »
   q_2: ┤ H ├─┤ Rz(1.3655) ├─────────────────■──────────────┼──────────────┼─────────────»
        ├───┤ ├────────────┤                                │ZZ(-0.15206)  │ZZ(-0.15202) »
   q_3: ┤ H ├─┤ Rz(1.3974) ├──■─────────────────────────────■──────────────■─────────────»
        ├───┤┌┴────────────┴┐ │ZZ(-0.15694)                                              »
   q_4: ┤ H

## Which plan has the lowest model score?

Enumerate the 64 allowed plans exactly, then compare that minimum with the ideal circuit and two local shot samples.

In [2]:
verification = json.loads((root / 'verification.json').read_text())
noise = json.loads((root / 'noisy-sample.json').read_text())
assert verification['fixtureSha256'] == noise['fixtureSha256'] == sha256(fixture)
assert verification['hardwareExecuted'] is False and noise['hardwareExecuted'] is False
ideal = analysis(case, verification['checks'][0]['idealMeasuredSimulation']['counts'])
noisy = analysis(case, noise['analysis']['counts'])
best, = ideal['bestAllowedPlans']
print('Lowest-score plan:', best)
print(f"Ideal probability: {ideal['idealBestAllowedPlanProbability']:.2%}")
for label, record in [('Ideal local sample', ideal), ('Noisy local sample', noisy)]:
    print(f"{label}: {record['counts'].get(best, 0):,} / {record['shots']:,} shots")

Lowest-score plan: 001101
Ideal probability: 51.80%
Ideal local sample: 1,079 / 2,048 shots
Noisy local sample: 824 / 2,048 shots


## IBM hardware evidence

The comparison concerns the measured distribution for one frozen decision. Scores are in model points.

In [3]:
record = json.loads((root / 'hardware.json').read_text())
assert record['fixtureSha256'] == sha256(fixture)
if record['status'] == 'recorded':
    assert record['hardwareExecuted'] and record['jobIdentityVerified']
    assert not record['dryRun'] and record['case'] == case['id']
    assert sha256(root / 'hardware' / 'compiled.qpy') == record['compiledSha256']
    measured = analysis(case, record['analysis']['counts'])
    print('Backend:', record['backend'], '| IBM job:', record['jobId'])
    print('Configured QPU cap (seconds):', record['maxExecutionSeconds'])
    print('IBM-reported QPU charge time (seconds):', record['qpuChargeSeconds'])
    print('IBM-reported resource units:', record['resourceUnits'])
    print(f"Best plan: {measured['counts'].get(best, 0):,} / {measured['shots']:,} shots"
          f" ({measured['observedBestAllowedPlanFrequency']:.2%})")
    print(f"Mean score: {measured['observedMeanCost']:.3f} model points")
    print(f"Total variation distance from ideal: {measured['totalVariationDistanceFromIdeal']:.4f}")
    print('Raw counts are in hardware.json.')
else:
    print('IBM hardware evidence pending.')

Backend: ibm_fez | IBM job: dbpr4klt7fqc7389seeg
Configured QPU cap (seconds): 60
IBM-reported QPU charge time (seconds): 3
IBM-reported resource units: 0.037228
Best plan: 886 / 2,048 shots (43.26%)
Mean score: 182.109 model points
Total variation distance from ideal: 0.0952
Raw counts are in hardware.json.
